# STAR testing - Hamilton STAR / STARLet

Validation notebook for the v1 STAR (`pylabrobot.hamilton.star`).

It drives a `STARDevice` - the instrument as a resource, with its deck as its child. The driver
stays reachable underneath as `star.driver`, and machine-level reads that the device does not
proxy are sent through it.

**`setup()` moves the machine.** Watch the log: it reports each phase at `DEBUG` and the machine
it found at `INFO`. It runs in three steps:

1. **discover** - read-only. Machine configuration, arm geometry, channel count, and every
   channel's firmware, width and installed hardware.
2. **initialize** - `C0 VI` on a machine that is not initialized, which homes every drive; or
   `C0 ZA` alone on one that is, to raise the channels to Z safety.
3. **capability bring-up** - the channels eject whatever is mounted on them, including grippers.

If you want to connect and look without anything moving, run `discover()` on its own; the cell
below shows how.

Set `protocol_mode` to `"simulation"` to run every cell against a simulated STAR - no hardware,
no USB, and the same code paths as the real driver.

## 1- Run identity

In [1]:
# --- Run identity ---
protocol_mode = "execution"  # simulation OR execution
user_name = "star_user"
run_identifier = "star_v1_validation"

# --- Which instrument ---
# One of the factories in pylabrobot.hamilton.star.device. It fixes the machine's footprint and
# where its deck sits inside it, and builds the matching deck.
instrument = "STAR"  # STAR OR STARLet OR STAR_with_extension_housing

# --- Device selection (only needed with more than one Hamilton on USB) ---
device_address = None  # USB address, e.g. 3
serial_number = None  # USB serial, e.g. "1234567"

# --- Motion ---
# The X-arm move near the end of this notebook only runs when this is True.
allow_x_arm_move = False

# --- 96-head ---
# Where the 96-head ejects when it is initialized: head channel A1, in deck mm. Initializing it
# throws off whatever is mounted, so this has to be somewhere tips may be dropped, which depends
# on where the waste sits on this deck - hence no default. Setup initializes the head when this is
# set, and reports that it cannot when it is None. This machine was last sent (-263.8, 108.3,
# 200.0), read off the `C0 EI` command in an earlier run.
head96_initialize_position = (-263.8, 108.3, 200.0)

## 2- Imports

In [2]:
from pylabrobot.hamilton.star.device import STAR, STAR_with_extension_housing, STARLet
from pylabrobot.hamilton.star.driver.features.head96 import Head96
from pylabrobot.hamilton.star.driver.master import STARDriver

## 3- Logging

Uses PyLabRobot's own `setup_logger`, exactly as every other PLR run does: a single
date-stamped file per day, appended to across runs. Both the file and the notebook are at
`IO` level, so every byte sent to and received from the machine is visible and recorded.

Re-running this cell is safe: `setup_logger` replaces the file handler and `verbose`
replaces the console handler, rather than stacking a second one of each.

In [3]:
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

log_dir = f"_logs/{protocol_mode}"

# PLR's own logger setup: one date-stamped file per day, appended to across runs. Re-running this
# cell replaces the file handler rather than stacking a second one, so lines are never duplicated.
pylabrobot.setup_logger(log_dir, level=LOG_LEVEL_IO)

# Console at IO level too: every byte sent and received appears in the notebook.
pylabrobot.verbose(True, level=LOG_LEVEL_IO)

print(f"appending to {log_dir}/pylabrobot-<YYYYMMDD>.log")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

appending to _logs/execution/pylabrobot-<YYYYMMDD>.log


2026-08-19 15:58:42,971 - pylabrobot - INFO - --- star_v1_validation (execution) ---


## 4- Connect and bring the machine up

In simulation this is a `STARSimulationDriver`, which answers as a real instrument does - the
same command assembly, error decoding and response parsing run either way.

Set `head96_initialize_position` above for setup to initialize the 96-head too; without it, setup
brings everything else up and reports that it could not do the head.

To connect **without moving anything**, replace `await star.setup()` with:

```python
await star._open()
star._connected = True
await star.discover()
```

In [4]:
build = {
  "STAR": STAR,
  "STARLet": STARLet,
  "STAR_with_extension_housing": STAR_with_extension_housing,
}[instrument]

# The instrument builds its own deck and hands it to the driver, which models the machine into it.
# A simulated one answers from that model, so it is built here rather than passed in.
if protocol_mode == "execution":
  star = build(driver=STARDriver(device_address=device_address, serial_number=serial_number))
else:
  star = build(simulation=True)

# Setup builds each capability the machine turns out to have, but not over one that is already
# there - so a capability configured here keeps its configuration. In simulation the head is
# already there and answers for itself, so configure that one rather than replacing it.
if head96_initialize_position is not None:
  if star.driver.head96 is None:
    star.driver.head96 = Head96(star.driver)
  star.head96.configuration.initialize_position = head96_initialize_position

await star.setup()

print(star)
# setup logs this summary at INFO; printed here too so it is the first thing you see.
print(star.driver.format_setup_summary())

2026-08-19 15:58:42,993 - pylabrobot.hamilton.star.driver.master - DEBUG - Setting up STAR on USB 0x08af:0x8000 ...
2026-08-19 15:58:42,995 - pylabrobot.io.usb - INFO - Finding USB device...
2026-08-19 15:58:43,016 - pylabrobot.io.usb - INFO - Found USB device.
2026-08-19 15:58:43,019 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-08-19 15:58:46,022 - pylabrobot.hamilton.star.drive

Hamilton STAR(STARDriver, 56-track deck)
[Hamilton STAR] Connected on USB 0x08af:0x8000
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Configuration: 54 slots
  Autoload: 1D barcode scanner
  Arms: 1
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 1340.2 mm, workspace -323.2 to 1517.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


In [5]:
deck = star.deck
deck

HamiltonSTARDeck(name='deck', location=Coordinate(121.800, 116.000, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

In [6]:
star.driver.deck

HamiltonSTARDeck(name='deck', location=Coordinate(121.800, 116.000, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

## 5- The rest of the configuration

What the setup summary does not already print, plus a check of this machine's firmware stack
against the stacks this driver has been driven with before.

In [7]:
from pylabrobot.hamilton.star.driver.confirmed_firmware_versions import suggest_entry, unconfirmed

c = star.driver.configuration
print(f"wash stations         : 1={c.wash_station_1_installed}  2={c.wash_station_2_installed}")
print(f"tip waste x           : {c.tip_waste_x_position} mm")
print(
  f"iSWAP collision-free  : {c.min_iswap_collision_free_position} to "
  f"{c.max_iswap_collision_free_position} mm"
)
print(f"pip maximal y         : {c.pip_maximal_y_position} mm")
print(f"initialized           : {await star.driver.request_initialization_status()}")

# Has each of this machine's boards been driven on the firmware it reports?
new = unconfirmed(star.driver.firmware)
print()
if not new:
  print(f"firmware: all {len(star.driver.firmware)} capabilities confirmed")
else:
  print(f"firmware: {len(new)} of {len(star.driver.firmware)} capabilities not seen before.")
  print("if this machine works, add them to confirmed_firmware_versions.py:")
  for capability, version in new.items():
    print(suggest_entry(capability, version))

2026-08-19 15:58:49,233 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QWid0065'


wash stations         : 1=False  2=False
tip waste x           : 1340.0 mm
iSWAP collision-free  : 350.0 to 1140.0 mm
pip maximal y         : 606.5 mm


2026-08-19 15:58:49,287 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QWid0065er00/00qw1')


initialized           : True

firmware: all 6 capabilities confirmed


## 6- The X-arms

A STAR always has a left arm and may have a right one. `star.x_arm` is the arm on a machine that
has only one, and refuses on a machine that has two.

In [8]:
for arm in (star.left_x_arm, star.right_x_arm):
  if arm is None:
    print("right: not installed")
    continue
  a = arm.configuration
  print(f"{arm.side:5s}: {a.model}   firmware {a.firmware_version}")
  print(f"       width {a.width} mm, travel {a.x_range} mm, workspace {a.workspace_range} mm")
  print(f"       wrap {a.wrap_size} mm, reference point: {a.reference_point}")
  print(
    f"       modules: pip={a.pip_installed} iswap={a.iswap_installed} "
    f"head96={a.head96_installed} xl={a.xl_channels_installed}"
  )

try:
  print(f"\nstar.x_arm -> {star.x_arm.side}")
except ValueError as e:
  print(f"\nstar.x_arm -> {e}")

left : hamilton_legacy_star_dual_rail_arm   firmware 1.4S 2012-04-25
       width 354.0 mm, travel (95.0, 1340.2) mm, workspace (-323.2, 1517.2) mm
       wrap 595.2 mm, reference point: center
       modules: pip=True iswap=True head96=True xl=False
right: not installed

star.x_arm -> left


## 7- The pipetting channels

`configuration` holds what every channel shares; `configuration.channels` holds one entry per
channel, read off the channel itself during discovery. A machine that reports no channels - a
96-head-only STAR - has no `star.pipettes` at all.

In [9]:
if star.pipettes is None:
  print("no channels installed")
else:
  p = star.pipettes.configuration
  print("shared by every channel:")
  print(f"  y drive   {p.y_drive_mm_per_increment} mm/increment")
  print(f"  z drive   {p.z_drive_mm_per_increment} mm/increment")
  print(f"  dispense  {p.dispensing_drive_uL_per_increment} uL/increment")
  print()
  print(
    f"{'ch':>3}  {'firmware':<20} {'width':>7}  {'channel':<12} {'head':<12} {'stop disc':<10} adc"
  )
  for i, ch in enumerate(p.channels):
    print(
      f"{i:>3}  {str(ch.firmware_version):<20} {str(ch.width):>7}  {str(ch.channel_type):<12} "
      f"{str(ch.head_type):<12} {str(ch.stop_disc_type):<10} {ch.pressure_adc}"
    )

shared by every channel:
  y drive   0.046302083 mm/increment
  z drive   0.01072765 mm/increment
  dispense  0.046876 uL/increment

 ch  firmware               width  channel      head         stop disc  adc
  0  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  1  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  2  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  3  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  4  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  5  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  6  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  7  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268


## 8- Sensor read: tip presence

Each channel's sleeve sensor reports whether a tip is mounted. This reads sensors; it does not
move a channel. After a full setup every channel should be empty: the channel initialization
ejects whatever was on them.

In [10]:
presence = await star.driver.request_tip_presence()
for channel, has_tip in enumerate(presence):
  print(f"  channel {channel}: {'tip' if has_tip else '-'}")

2026-08-19 15:58:49,333 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RTid0066'
2026-08-19 15:58:49,375 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RTid0066er00/00rt0 0 0 0 0 0 0 0')


  channel 0: -
  channel 1: -
  channel 2: -
  channel 3: -
  channel 4: -
  channel 5: -
  channel 6: -
  channel 7: -


## 9- The front cover

Two read-only commands, and what they mean is exactly what this check is for.

`C0 RW` reports three inputs, the first of them the cover input. `C0 QC` reports the cover
position. Neither says whether a cover is *fitted*: the master acts only on its non-volatile
configuration, so `main_front_cover_monitoring_installed` is what decides whether the cover is
watched at all, and `star.front_cover` exists only when it is set.

This machine reports it as not installed while the cover and its switch are physically there, so
`QC` is sent raw below rather than through the capability.

**Run this three times and record what changes**: cover shut, cover open, and cover cable
disconnected. If the cover input tracks the position it is a position input; if it holds while
the position changes it is a presence input; if neither moves, the master is not reading the
switch at all - which is what a configuration that says the monitoring is not installed predicts.

That last outcome is the one that decides whether `FrontCover` is worth keeping: a machine that
answers nothing here has no cover to drive, and the capability would only ever be an empty
`request_position` on machines configured differently from this one.


In [11]:
cover_input, second_input, reserve_input = await star.driver.request_cover_input_status()
print(f"inputs        : cover={cover_input}  second={second_input}  reserve={reserve_input}")

c = star.driver.configuration
print(
  f"monitoring    : main={c.main_front_cover_monitoring_installed}"
  f"  additional={c.additional_front_cover_monitoring_installed}"
)
print(f"covers        : left={c.left_cover_installed}  right={c.right_cover_installed}")
print(f"capability    : {star.front_cover}")

# C0 QC - request cover position. Read-only, and sent raw so it answers even on a machine whose
# configuration says the monitoring is not installed.
print(f"position (raw): {await star.driver.send_raw_command('C0QCid9989')}")
if star.front_cover is not None:
  print(f"position      : {await star.front_cover.request_position()}")

2026-08-19 15:58:49,385 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RWid0067'
2026-08-19 15:58:49,408 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RWid0067er00/00rw000')
2026-08-19 15:58:49,412 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QCid9989'


inputs        : cover=False  second=False  reserve=False
monitoring    : main=False  additional=False
covers        : left=False  right=False
capability    : None


2026-08-19 15:58:49,432 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QCid9989er00/00qc1')


position (raw): C0QCid9989er00/00qc1


## 10- Move the X-arm

**This moves the arm and everything mounted on it.** Only run it with the deck clear along the
path, and only after setup has raised the channels to Z safety.

Gated on `allow_x_arm_move`, set at the top of the notebook.

In [12]:
arm = star.x_arm
print(f"travel range: {arm.configuration.x_range} mm")

target = 500.0
if allow_x_arm_move:
  await arm.move_x(target)
  print(f"moved to {target} mm")
else:
  print(f"skipped. set allow_x_arm_move = True to move to {target} mm")

# out-of-range targets are refused before anything reaches the wire
try:
  await arm.move_x(5000.0)
except ValueError as e:
  print("guard:", e)

travel range: (95.0, 1340.2) mm
skipped. set allow_x_arm_move = True to move to 500.0 mm
guard: left X-arm x=5000.0mm is outside its drive travel range [95.0, 1340.2].


In [13]:
deck.get_resource("left_x_arm").get_location_wrt(deck)

Coordinate(x=523.0, y=0.0, z=334.7)

In [14]:
await star.x_arm.request_position(), deck.get_resource("left_x_arm").get_location_wrt(deck)

2026-08-19 15:58:49,468 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0068'
2026-08-19 15:58:49,481 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0068rx+07000 +000070000')


(700.0, Coordinate(x=523.0, y=0.0, z=334.7))

In [15]:
(
  await star.driver.send_command(module="C0", command="RX"),
  await star.driver.send_command(module="C0", command="QX"),
)

2026-08-19 15:58:49,496 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RXid0069'
2026-08-19 15:58:49,517 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RXid0069er00/00rx+07000')
2026-08-19 15:58:49,520 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QXid0070'
2026-08-19 15:58:49,541 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QXid0070er00/00rx+00000')


('C0RXid0069er00/00rx+07000', 'C0QXid0070er00/00rx+00000')

In [16]:
# The same gate as the section above: this moves the arm.
allow_x_arm_move = True
if allow_x_arm_move:
  await star.x_arm.move_x(500.0)
  print("moved to 500.0 mm")
else:
  print("skipped. set allow_x_arm_move = True to move")

# What the machine says, and where the model puts the arm's reference point. They should agree.
position = await star.x_arm.request_position()
arm_resource = deck.get_resource("left_x_arm")
seated = arm_resource.get_location_wrt(deck)
print(f"machine: {position} mm")
print(f"model  : {seated.x + arm_resource.get_anchor(x=star.x_arm.reference_anchor).x} mm")

2026-08-19 15:58:49,553 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0071la05000lr3lw7'
2026-08-19 15:58:51,068 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0071er00')
2026-08-19 15:58:51,073 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0072'
2026-08-19 15:58:51,083 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0072rx+05002 +000050017')
2026-08-19 15:58:51,086 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.2 mm
2026-08-19 15:58:51,090 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0073'
2026-08-19 15:58:51,098 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0073rx+05001 +000050013')


moved to 500.0 mm
machine: 500.1 mm
model  : 500.1 mm


In [17]:
deck.get_resource("left_x_arm").get_size_x() / 2

177.0

## 11- Who closes the loop on an X move?

`X0 XP` asked for 500.0 mm and the arm came to rest at 499.8 - two increments short. The master has
its own absolute move for the same axis, `C0 JX`, and a variant that raises everything to Z safety
first, `C0 KX`. If the master runs a control loop to the target, its move should land closer than
the board's.

**This moves the arm.** Gated on `allow_x_arm_move`. It drives to each target twice, once through
each command, reading back where the arm actually stopped, and reports the residual.


In [18]:
targets = [500.0, 800.0, 1_000.0]

if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
else:
  print(f"{'target':>8} {'X0 XP':>10} {'C0 JX':>10}   residual, mm")
  for target in targets:
    increments = f"{round(target * 10):05}"

    await star.x_arm.move_x(target)  # X0 XP, and it reads back
    by_board = await star.x_arm.request_position()

    # move away first, so the second command has the same distance to cover as the first did not
    await star.x_arm.move_x(target - 50.0)
    await star.driver.send_command(module="C0", command="JX", xs=increments)
    by_master = await star.x_arm.request_position()

    print(f"{target:8.1f} {by_board - target:10.2f} {by_master - target:10.2f}")

  target      X0 XP      C0 JX   residual, mm


2026-08-19 15:58:51,141 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0074la05000lr3lw7'
2026-08-19 15:58:51,353 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0074er00')
2026-08-19 15:58:51,359 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0075'
2026-08-19 15:58:51,367 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0075rx+05001 +000050005')
2026-08-19 15:58:51,371 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.1 mm
2026-08-19 15:58:51,374 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0076'
2026-08-19 15:58:51,382 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0076rx+05001 +000050005')
2026-08-19 15:58:51,386 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0077la04500lr3lw7'
2026-08-19 15:58:52,298 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0077er00')
2026-08-19 1

   500.0       0.10      -0.20


2026-08-19 15:58:54,926 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0081er00')
2026-08-19 15:58:54,930 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0082'
2026-08-19 15:58:54,940 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0082rx+07998 +000079981')
2026-08-19 15:58:54,944 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 800.0 mm and came to rest at 799.8 mm
2026-08-19 15:58:54,947 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0083'
2026-08-19 15:58:54,956 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0083rx+07999 +000079988')
2026-08-19 15:58:54,960 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0084la07500lr3lw7'
2026-08-19 15:58:55,922 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0084er00')
2026-08-19 15:58:55,927 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0085'
2026-08-19 15:58:55,936 -

   800.0      -0.10      -0.20


2026-08-19 15:58:58,407 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0088er00')
2026-08-19 15:58:58,412 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0089'
2026-08-19 15:58:58,421 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0089rx+09999 +000099991')
2026-08-19 15:58:58,425 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 1000.0 mm and came to rest at 999.9 mm
2026-08-19 15:58:58,428 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0090'
2026-08-19 15:58:58,436 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0090rx+09999 +000099992')
2026-08-19 15:58:58,440 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0091la09500lr3lw7'
2026-08-19 15:58:59,354 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0091er00')
2026-08-19 15:58:59,359 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0092'
2026-08-19 15:58:59,368 

  1000.0      -0.10      -0.20


## 12- The autoload's X resolution

The driver reads the scanner X drive as 0.1 mm per increment. This machine has no `I0 QU` to ask
(`er30`, unknown command) and `I0 RA raau` answers `au0`, which the mapping calls 0.1 mm/step - so
the value is inferred, not measured.

Tracks are 22.5 mm apart, so moving between two tracks and reading the position each time measures
it directly: a unit that is really 0.125 mm/step would report the same travel 25% wide.

**This moves the autoload.** It travels along the front of the deck; nothing else moves.


In [19]:
if star.autoload is None:
  print("no autoload on this machine")
else:
  first, second = 10, 20
  await star.autoload.move_to_track(first)
  x_first = await star.autoload.request_x_position()
  await star.autoload.move_to_track(second)
  x_second = await star.autoload.request_x_position()

  tracks = second - first
  measured = (x_second - x_first) / tracks
  print(f"track {first} at {x_first:.2f} mm, track {second} at {x_second:.2f} mm")
  print(f"measured {measured:.3f} mm per track, against 22.5 mm expected")
  print(f"so the drive resolution is {0.1 * 22.5 / measured:.4f} mm per increment, read as 0.1")

  await star.autoload.park()

2026-08-19 15:59:00,290 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RZid0095'
2026-08-19 15:59:00,306 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RZid0095rz+0000 +0000')
2026-08-19 15:59:00,315 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0XPid0096xp10xv2500xr3xw7'
2026-08-19 15:59:04,672 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0XPid0096er00')
2026-08-19 15:59:04,677 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RXid0097'
2026-08-19 15:59:04,686 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RXid0097rx+02025 +02025')
2026-08-19 15:59:04,691 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RZid0098'
2026-08-19 15:59:04,701 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RZid0098rz+0000 +0000')
2026-08-19 15:59:04,705 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0XPid0099xp20xv2500xr3xw7'
2026-08-19 15:59:06,011 - pylabrobot.io.usb - IO - [0x8af:0x8000

track 10 at 202.50 mm, track 20 at 427.50 mm
measured 22.500 mm per track, against 22.5 mm expected
so the drive resolution is 0.1000 mm per increment, read as 0.1


2026-08-19 15:59:09,500 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0XPid0102er00')


## 13- Can we close the loop ourselves?

`X0 XP` lands within 0.3 mm and `C0 JX` is consistently 0.3 mm short, so neither command closes a
position loop. The question is whether we can: does re-commanding move the arm at all, and does
correcting by the measured error converge?

Two passes per target. The first re-sends the same target and watches whether anything changes; the
second commands `target + error`, which should walk the residual out if the drive responds to small
corrections at all. Note the read itself jitters by one increment, so 0.1 mm is the noise floor and
a tolerance below that will never be met.

**This moves the arm.** Gated on `allow_x_arm_move`.


In [20]:
tolerance = 0.15  # mm, above the read's own one-increment jitter
max_attempts = 4

if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
else:
  for target in (500.0, 800.0):
    await star.x_arm.move_x(target - 50.0)  # approach from the same side every time
    await star.x_arm.move_x(target)
    print(f"\ntarget {target} mm")

    print("  re-sending the same target:")
    for attempt in range(max_attempts):
      reached = await star.x_arm.request_position()
      print(f"    attempt {attempt}: at {reached:.1f} mm, error {reached - target:+.1f} mm")
      if abs(reached - target) <= tolerance:
        break
      await star.x_arm.move_x(target)

    await star.x_arm.move_x(target - 50.0)
    await star.x_arm.move_x(target)
    print("  correcting by the measured error:")
    for attempt in range(max_attempts):
      reached = await star.x_arm.request_position()
      error = target - reached
      print(f"    attempt {attempt}: at {reached:.1f} mm, error {-error:+.1f} mm")
      if abs(error) <= tolerance:
        break
      await star.x_arm.move_x(target + error)

2026-08-19 15:59:09,511 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0103la04500lr3lw7'
2026-08-19 15:59:11,521 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0103er00')
2026-08-19 15:59:11,526 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0104'
2026-08-19 15:59:11,536 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0104rx+04501 +000045008')
2026-08-19 15:59:11,539 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 450.0 mm and came to rest at 450.1 mm
2026-08-19 15:59:11,542 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0105la05000lr3lw7'
2026-08-19 15:59:12,455 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0105er00')
2026-08-19 15:59:12,461 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0106'
2026-08-19 15:59:12,470 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0106rx+04997 +000049974')
2026-08-19 1


target 500.0 mm
  re-sending the same target:
    attempt 0: at 499.8 mm, error -0.2 mm


2026-08-19 15:59:12,700 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0108er00')
2026-08-19 15:59:12,705 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0109'
2026-08-19 15:59:12,715 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0109rx+04999 +000049994')
2026-08-19 15:59:12,718 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 499.9 mm
2026-08-19 15:59:12,721 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0110'
2026-08-19 15:59:12,730 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0110rx+05000 +000049996')
2026-08-19 15:59:12,734 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0111la04500lr3lw7'


    attempt 1: at 500.0 mm, error +0.0 mm


2026-08-19 15:59:13,698 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0111er00')
2026-08-19 15:59:13,703 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0112'
2026-08-19 15:59:13,712 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0112rx+04501 +000045013')
2026-08-19 15:59:13,715 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 450.0 mm and came to rest at 450.1 mm
2026-08-19 15:59:13,718 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0113la05000lr3lw7'
2026-08-19 15:59:14,680 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0113er00')
2026-08-19 15:59:14,686 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0114'
2026-08-19 15:59:14,695 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0114rx+04999 +000049987')
2026-08-19 15:59:14,698 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 

...

2026-08-19 15:59:19,823 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0129er00')
2026-08-19 15:59:19,827 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0130'
2026-08-19 15:59:19,836 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0130rx+07996 +000079960')
2026-08-19 15:59:19,839 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 799.7 mm and came to rest at 799.6 mm
2026-08-19 15:59:19,842 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0131'
2026-08-19 15:59:19,850 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0131rx+07996 +000079961')
2026-08-19 15:59:19,854 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0132la08004lr3lw7'


    attempt 2: at 799.6 mm, error -0.4 mm


2026-08-19 15:59:20,117 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0132er00')
2026-08-19 15:59:20,121 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0133'
2026-08-19 15:59:20,131 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0133rx+08005 +000080054')
2026-08-19 15:59:20,135 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 800.4 mm and came to rest at 800.5 mm
2026-08-19 15:59:20,137 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0134'
2026-08-19 15:59:20,147 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0134rx+08005 +000080052')
2026-08-19 15:59:20,151 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0135la07995lr3lw7'


    attempt 3: at 800.5 mm, error +0.5 mm


2026-08-19 15:59:20,461 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0135er00')
2026-08-19 15:59:20,465 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0136'
2026-08-19 15:59:20,474 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0136rx+07994 +000079944')
2026-08-19 15:59:20,477 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 799.5 mm and came to rest at 799.4 mm


## 13b- Why is each X move short: undershoot, or an offset?

`X0 XP` lands 0.2 mm short of a long move and exactly on a short one, so re-sending the target
converges. `C0 JX` was short by 0.30 mm at all three targets, which is too consistent for
deceleration undershoot - but it was only ever approached from below, so the two explanations were
never separated.

They differ in how the error is signed.

| | error follows | approached from below | approached from above | from rest |
|---|---|---|---|---|
| undershoot | the direction of travel | lands low | lands high | no error |
| offset | the axis, always the same way | lands low | lands low | lands low |

So the experiment is the same target reached from both sides, and from rest, through each command.
Distance is varied too, since undershoot grows with speed and a short move showed none.

**This moves the arm.** Gated on `allow_x_arm_move`. Every start position is settled with `X0 XP`,
re-sent until it is within tolerance, so each measurement begins from a known place.


In [21]:
target = 600.0
distances = (0.5, 100.0)
tolerance = 0.15

if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
else:

  async def at() -> float:
    return await star.x_arm.request_position()

  async def settle_at(x: float) -> float:
    """Put the arm on x with the command that converges, and say where it ended up."""
    for _ in range(4):
      await star.x_arm.move_x(x)
      if abs(await at() - x) <= tolerance:
        break
    return await at()

  async def send(command: str, x: float) -> None:
    if command == "X0 XP":
      await star.x_arm.move_x(x)
    else:
      await star.driver.send_command(module="C0", command="JX", xs=f"{round(x * 10):05}")

  print(f"{'command':8s} {'from':>8} {'distance':>9} {'started':>9} {'reached':>9} {'error':>7}")
  for command in ("X0 XP", "C0 JX"):
    for distance in distances:
      for direction, start in (("below", target - distance), ("above", target + distance)):
        started = await settle_at(start)
        await send(command, target)
        reached = await at()
        print(
          f"{command:8s} {direction:>8} {distance:9.1f} {started:9.1f} {reached:9.1f}"
          f" {reached - target:+7.1f}"
        )
    # from rest on the target itself: undershoot has nothing to undershoot
    started = await settle_at(target)
    await send(command, target)
    reached = await at()
    print(
      f"{command:8s} {'rest':>8} {0.0:9.1f} {started:9.1f} {reached:9.1f} {reached - target:+7.1f}"
    )

  print("\nre-sending the same target, through each command:")
  for command in ("X0 XP", "C0 JX"):
    await settle_at(target - 100.0)
    print(f"  {command}")
    for attempt in range(4):
      await send(command, target)
      reached = await at()
      print(f"    attempt {attempt}: at {reached:9.1f} mm, error {reached - target:+.1f} mm")
      if abs(reached - target) <= tolerance:
        break

command      from  distance   started   reached   error


2026-08-19 15:59:20,513 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0137la05995lr3lw7'
2026-08-19 15:59:22,076 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0137er00')
2026-08-19 15:59:22,082 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0138'
2026-08-19 15:59:22,091 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0138rx+05996 +000059961')
2026-08-19 15:59:22,094 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 599.5 mm and came to rest at 599.6 mm
2026-08-19 15:59:22,097 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0139'
2026-08-19 15:59:22,105 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0139rx+05996 +000059958')
2026-08-19 15:59:22,109 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0140'
2026-08-19 15:59:22,118 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0140rx+05996 +000059956')
2026-08-19

X0 XP       below       0.5     599.6     600.1    +0.1


2026-08-19 15:59:22,681 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0144er00')
2026-08-19 15:59:22,686 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0145'
2026-08-19 15:59:22,695 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0145rx+06006 +000060059')
2026-08-19 15:59:22,699 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 600.5 mm and came to rest at 600.6 mm
2026-08-19 15:59:22,701 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0146'
2026-08-19 15:59:22,710 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0146rx+06006 +000060058')
2026-08-19 15:59:22,714 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0147'
2026-08-19 15:59:22,723 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0147rx+06006 +000060058')
2026-08-19 15:59:22,727 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0148la06000lr3lw7'
2026-08-19

X0 XP       above       0.5     600.6     600.0    +0.0


...

    attempt 0: at     599.9 mm, error -0.1 mm


2026-08-19 15:59:39,514 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0209er00')
2026-08-19 15:59:39,518 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0210'
2026-08-19 15:59:39,528 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0210rx+05001 +000050011')
2026-08-19 15:59:39,530 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.1 mm
2026-08-19 15:59:39,533 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0211'
2026-08-19 15:59:39,541 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0211rx+05001 +000050006')
2026-08-19 15:59:39,544 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0212'
2026-08-19 15:59:39,554 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0212rx+05000 +000050004')
2026-08-19 15:59:39,558 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0JXid0213xs06000'


  C0 JX


2026-08-19 15:59:41,139 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0JXid0213er00/00')
2026-08-19 15:59:41,144 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0214'
2026-08-19 15:59:41,153 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0214rx+06000 +000059998')


    attempt 0: at     600.0 mm, error +0.0 mm


## 13c- Why did the same command measure differently?

Section 11 found `C0 JX` short by 0.30 mm at 500, 800 and 1000 mm. Section 13b found it accurate at
600 mm from either side, at both 0.5 and 100 mm, and from rest. Both read the position 3-6 ms after
a reply that only comes once the move is over, so neither was reading a moving arm.

Two differences remain between the protocols, and this separates them.

- **Section 11 moved exactly 50 mm every time.** 13b used 0.5 mm and 100 mm.
- **Section 11 started where `X0 XP` had just left the arm**, which is itself 0.2 mm off target,
  rather than from a settled position.

So each target is reached by `JX` twice: once from a start `XP` left behind, exactly as section 11
did, and once from the same start settled to the millimetre. If only the first is short, what
matters is the state `XP` leaves; if both are, it is the 50 mm distance.

**This moves the arm.** Gated on `allow_x_arm_move`.


In [22]:
if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
else:

  async def at() -> float:
    return await star.x_arm.request_position()

  async def settle_at(x: float) -> float:
    for _ in range(4):
      await star.x_arm.move_x(x)
      if abs(await at() - x) <= 0.15:
        break
    return await at()

  async def jump_to(x: float) -> None:
    await star.driver.send_command(module="C0", command="JX", xs=f"{round(x * 10):05}")

  print(f"{'target':>8} {'start':>22} {'started':>9} {'reached':>9} {'error':>7}")
  for target in (500.0, 800.0, 1_000.0):
    # exactly as section 11: one XP move to 50 mm below, wherever that lands
    await star.x_arm.move_x(target)
    await star.x_arm.move_x(target - 50.0)
    started = await at()
    await jump_to(target)
    reached = await at()
    print(
      f"{target:8.1f} {'as XP left it':>22} {started:9.1f} {reached:9.1f} {reached - target:+7.1f}"
    )

    # the same start, settled first
    started = await settle_at(target - 50.0)
    await jump_to(target)
    reached = await at()
    print(f"{target:8.1f} {'settled':>22} {started:9.1f} {reached:9.1f} {reached - target:+7.1f}")

2026-08-19 15:59:41,177 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0215la05000lr3lw7'


  target                  start   started   reached   error


2026-08-19 15:59:42,387 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0215er00')
2026-08-19 15:59:42,392 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0216'
2026-08-19 15:59:42,401 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0216rx+05001 +000050010')
2026-08-19 15:59:42,405 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.1 mm
2026-08-19 15:59:42,408 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0217la04500lr3lw7'
2026-08-19 15:59:43,366 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0217er00')
2026-08-19 15:59:43,371 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0218'
2026-08-19 15:59:43,380 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0218rx+04501 +000045014')
2026-08-19 15:59:43,383 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 450.0 

   500.0          as XP left it     450.1     499.9    -0.1


2026-08-19 15:59:45,260 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0222er00')
2026-08-19 15:59:45,265 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0223'
2026-08-19 15:59:45,276 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0223rx+04502 +000045015')
2026-08-19 15:59:45,278 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 450.0 mm and came to rest at 450.2 mm
2026-08-19 15:59:45,281 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0224'
2026-08-19 15:59:45,290 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0224rx+04501 +000045011')
2026-08-19 15:59:45,293 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0225'
2026-08-19 15:59:45,303 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0225rx+04501 +000045007')
2026-08-19 15:59:45,306 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0JXid0226xs05000'
2026-08-19 15:59

...

   800.0                settled     750.1     800.0    +0.0


2026-08-19 15:59:53,901 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0241er00')
2026-08-19 15:59:53,906 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0242'
2026-08-19 15:59:53,916 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0242rx+09999 +000099992')
2026-08-19 15:59:53,919 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 1000.0 mm and came to rest at 999.9 mm
2026-08-19 15:59:53,922 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0243la09500lr3lw7'
2026-08-19 15:59:54,834 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0243er00')
2026-08-19 15:59:54,839 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0244'
2026-08-19 15:59:54,851 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0244rx+09503 +000095025')
2026-08-19 15:59:54,854 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 950.0

  1000.0          as XP left it     950.2     999.8    -0.2


2026-08-19 15:59:56,735 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0248er00')
2026-08-19 15:59:56,740 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0249'
2026-08-19 15:59:56,750 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0249rx+09501 +000095012')
2026-08-19 15:59:56,753 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 950.0 mm and came to rest at 950.1 mm
2026-08-19 15:59:56,756 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0250'
2026-08-19 15:59:56,764 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0250rx+09501 +000095009')
2026-08-19 15:59:56,768 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0251'
2026-08-19 15:59:56,778 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0251rx+09501 +000095008')
2026-08-19 15:59:56,782 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0JXid0252xs10000'
2026-08-19 15:59

  1000.0                settled     950.1    1000.0    +0.0


## 13d- Do the master and the drive board agree on where the arm is?

`C0 JX` lands 0.3 mm short when the arm sits where `X0 XP` left it, and exactly on target when the
same start is settled first. Same command, same target, same 50 mm - so it is neither distance nor
direction, and the reply only arrives once the move is over.

What is left is the frame. `JX` is the master's command and `X0 RX` is the board's read, and the two
keep their own idea of where the arm is. If they part company exactly when `XP` has just stopped
short, then `JX` is accurate in the master's frame and every error measured here is the gap between
the frames, not a positioning error at all.

So: ask both, at the same moment, in both states.

`C0 RX` reads the master's position and `X0 RX` the board's, and both are read-only. Only the two
positioning moves touch the machine, and they are gated on `allow_x_arm_move`.


In [23]:
if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
else:
  import re

  async def both_frames() -> tuple:
    """The master's position and the board's, read back to back."""
    master = await star.driver.send_command(module="C0", command="RX")
    board = await star.driver.send_command(module="X0", command="RX")
    return (
      int(re.search(r"rx([+-]\d+)", master).group(1)) / 10,
      int(re.search(r"rx\s*([+-]\d+)", board).group(1)) / 10,
    )

  target, start = 500.0, 450.0

  # where XP leaves it: one move down, wherever it lands
  await star.x_arm.move_x(target)
  await star.x_arm.move_x(start)
  master, board = await both_frames()
  print(f"as XP left it: master {master:8.1f}   board {board:8.1f}   apart {master - board:+.1f}")
  await star.driver.send_command(module="C0", command="JX", xs=f"{round(target * 10):05}")
  master, board = await both_frames()
  print(
    f"  after JX to {target}: master {master:8.1f}   board {board:8.1f}   apart {master - board:+.1f}"
  )

  # the same start, settled
  for _ in range(4):
    await star.x_arm.move_x(start)
    if abs(await star.x_arm.request_position() - start) <= 0.15:
      break
  master, board = await both_frames()
  print(f"\nsettled      : master {master:8.1f}   board {board:8.1f}   apart {master - board:+.1f}")
  await star.driver.send_command(module="C0", command="JX", xs=f"{round(target * 10):05}")
  master, board = await both_frames()
  print(
    f"  after JX to {target}: master {master:8.1f}   board {board:8.1f}   apart {master - board:+.1f}"
  )

2026-08-19 15:59:58,017 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0254la05000lr3lw7'
2026-08-19 15:59:59,973 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0254er00')
2026-08-19 15:59:59,978 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0255'
2026-08-19 15:59:59,988 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0255rx+05001 +000050008')
2026-08-19 15:59:59,991 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.1 mm
2026-08-19 15:59:59,993 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0256la04500lr3lw7'
2026-08-19 16:00:00,906 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0256er00')
2026-08-19 16:00:00,911 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0257'
2026-08-19 16:00:00,920 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0257rx+04503 +000045027')
2026-08-19 1

as XP left it: master    450.2   board    450.2   apart +0.0


2026-08-19 16:00:01,851 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0JXid0260er00/00')
2026-08-19 16:00:01,856 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RXid0261'
2026-08-19 16:00:01,877 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RXid0261er00/00rx+04999')
2026-08-19 16:00:01,882 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0262'
2026-08-19 16:00:01,892 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0262rx+05000 +000050000')
2026-08-19 16:00:01,896 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0263la04500lr3lw7'


  after JX to 500.0: master    499.9   board    500.0   apart -0.1


2026-08-19 16:00:02,854 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0263er00')
2026-08-19 16:00:02,857 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0264'
2026-08-19 16:00:02,867 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0264rx+04502 +000045015')
2026-08-19 16:00:02,869 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 450.0 mm and came to rest at 450.2 mm
2026-08-19 16:00:02,870 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0265'
2026-08-19 16:00:02,880 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0265rx+04501 +000045011')
2026-08-19 16:00:02,884 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RXid0266'
2026-08-19 16:00:02,905 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RXid0266er00/00rx+04501')
2026-08-19 16:00:02,908 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0267'
2026-08-19 16:00:02,918 - p


settled      : master    450.1   board    450.0   apart +0.1


2026-08-19 16:00:04,105 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0JXid0268er00/00')
2026-08-19 16:00:04,110 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RXid0269'
2026-08-19 16:00:04,131 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RXid0269er00/00rx+05000')
2026-08-19 16:00:04,136 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0270'
2026-08-19 16:00:04,145 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0270rx+05000 +000049997')


  after JX to 500.0: master    500.0   board    500.0   apart +0.0


## 13e- Does the arm ring down, and does the 96-head's Y position change it?

Two consecutive position reads after a move returned 499.7 then 499.8, so the arm was still moving
while it was being measured. That would explain everything seen so far without any positioning
error: the reply arrives when the move ends, not when the arm stops.

If it is ring-down, the 96-head's Y position should matter. The head is a mass cantilevered off an
arm that is known to wobble, and a head parked forward has more leverage than one parked back. A
gentler acceleration should matter too, and so should simply waiting.

So this sweeps three things and reports both frames throughout:

- the 96-head at the front, middle and back of its Y travel (`H0 YA`, which moves the head's own Y
  drive and nothing else),
- the arm's acceleration index from gentlest to hardest,
- the position immediately after the move, polled for a second, and again after a wait.

`drift` is how far the reading moved on its own after the move ended. If it is around 0.2 mm and
grows with a forward head or a harder acceleration, the arm is ringing and the answer is to wait
rather than to re-command.

**This moves the arm and the 96-head.** Gated on `allow_x_arm_move`, and the deck must be clear.


In [24]:
import asyncio
import re
import time

start, target = 500.0, 600.0
accelerations = (1, 3, 5)
poll_for = 1.0  # seconds
wait_after = 1.0

if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
elif star.head96 is None:
  print("no 96-head on this machine")
else:

  async def frames() -> tuple:
    """The master's position and the board's, in mm."""
    master = await star.driver.send_command(module="C0", command="RX")
    board = await star.driver.send_command(module="X0", command="RX")
    return (
      int(re.search(r"rx([+-]\d+)", master).group(1)) / 10,
      int(re.search(r"rx\s*([+-]\d+)", board).group(1)) / 10,
    )

  async def move_head_y(y: float) -> None:
    increments = star.head96.configuration.y_drive_mm_to_increments(y)
    await star.driver.send_command(
      module="H0", command="YA", ya=f"{increments:05}", read_timeout=30
    )

  async def settle_at(x: float) -> None:
    for _ in range(4):
      await star.x_arm.move_x(x)
      if abs(await star.x_arm.request_position() - x) <= 0.15:
        break

  y_low, y_high = star.head96.configuration.y_range
  span = y_high - y_low
  # A quarter in from each end: the drive takes the full command range but the machine refuses the
  # extremes as outside its permitted area, which section 13f measures.
  head_positions = {
    "front": y_low + span / 4,
    "middle": (y_low + y_high) / 2,
    "back": y_high - span / 4,
  }
  print(f"96-head Y travel: {y_low} to {y_high} mm; sampling {list(head_positions.values())}")
  print(
    f"\n{'head Y':>8} {'accel':>6} {'first':>16} {'after poll':>16} {'after wait':>16} {'drift':>7}"
  )

  for where, y in head_positions.items():
    try:
      await star.head96.move_y(y)
    except Exception as refused:  # noqa: BLE001 - the machine says what it will not do
      print(f"{where:>8} {'-':>6} refused at y={y}: {refused}")
      continue
    for acceleration in accelerations:
      await settle_at(start)
      await star.driver.send_command(
        module="X0", command="XP", la=f"{round(target * 10):05}", lr=f"{acceleration:01}", lw="7"
      )
      first = await frames()

      deadline = time.monotonic() + poll_for
      polled = first
      while time.monotonic() < deadline:
        polled = await frames()
      await asyncio.sleep(wait_after)
      waited = await frames()

      print(
        f"{where:>8} {acceleration:>6} {first[0]:7.1f}/{first[1]:<8.1f}"
        f" {polled[0]:7.1f}/{polled[1]:<8.1f} {waited[0]:7.1f}/{waited[1]:<8.1f}"
        f" {waited[1] - first[1]:+7.1f}"
      )
  print(
    "\ncolumns are master/board, in mm; drift is board after the wait minus board at first read"
  )

96-head Y travel: 93.75 to 562.5 mm; sampling [210.9375, 328.125, 445.3125]

  head Y  accel            first       after poll       after wait   drift


2026-08-19 16:00:04,185 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0271rayv'
2026-08-19 16:00:04,194 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0271yv25000')
2026-08-19 16:00:04,198 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0272rayr'
2026-08-19 16:00:04,208 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0272yr35000')
2026-08-19 16:00:04,211 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0YAid0273ya13500yv25000yr35000yw15'
2026-08-19 16:00:05,875 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0YAid0273er00')
2026-08-19 16:00:05,880 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0274'
2026-08-19 16:00:05,889 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0274ry+13500 +13506')
2026-08-19 16:00:05,893 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0275la05000lr3lw7'
2026-08-19 16:00:06,060 - pylabrobot.io.usb - IO - [0x8af:0x80

   front      1   600.0/600.0      600.0/600.0      600.0/600.0       +0.0


2026-08-19 16:00:10,956 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0337er00')
2026-08-19 16:00:10,960 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0338'
2026-08-19 16:00:10,970 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0338rx+05002 +000050019')
2026-08-19 16:00:10,973 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.2 mm
2026-08-19 16:00:10,974 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0339'
2026-08-19 16:00:10,984 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0339rx+05001 +000050014')
2026-08-19 16:00:10,987 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0340la06000lr3lw7'
2026-08-19 16:00:12,199 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0340er00')
2026-08-19 16:00:12,204 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RXid0341'
2026-08-19 16:00:12,227 -

   front      3   599.8/599.9      600.0/600.0      600.0/600.0       +0.1


...

    back      1   600.0/600.0      600.0/600.0      600.0/600.0       +0.0


2026-08-19 16:00:40,701 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0723er00')
2026-08-19 16:00:40,705 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0724'
2026-08-19 16:00:40,715 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0724rx+05001 +000050013')
2026-08-19 16:00:40,718 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.1 mm
2026-08-19 16:00:40,721 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0725'
2026-08-19 16:00:40,729 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0725rx+05001 +000050007')
2026-08-19 16:00:40,733 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0726la06000lr3lw7'
2026-08-19 16:00:41,946 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0726er00')
2026-08-19 16:00:41,950 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RXid0727'
2026-08-19 16:00:41,971 -

    back      3   599.9/600.0      600.0/600.0      600.0/600.0       +0.0


2026-08-19 16:00:45,238 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0785er00')
2026-08-19 16:00:45,242 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0786'
2026-08-19 16:00:45,252 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0786rx+05001 +000050013')
2026-08-19 16:00:45,255 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.1 mm
2026-08-19 16:00:45,256 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0787'
2026-08-19 16:00:45,266 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0787rx+05001 +000050008')
2026-08-19 16:00:45,269 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0788la06000lr5lw7'
2026-08-19 16:00:46,283 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0788er00')
2026-08-19 16:00:46,287 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RXid0789'
2026-08-19 16:00:46,309 -

    back      5   599.8/599.9      600.0/600.0      600.0/600.0       +0.1

columns are master/board, in mm; drift is board after the wait minus board at first read


## 13f- Where does the 96-head's permitted Y area actually start?

`H0 YA` takes 6000 to 36000 increments - 93.75 to 562.5 mm - in every firmware document from 2013
on. This machine refuses the low end with error 58, "Y drive position outside of permitted area",
so what the parameter accepts and what the machine allows are two different things. The head shares
the arm with the pipetting channels, which is the obvious thing that would constrain it.

This finds the front limit by bisection: a position the machine takes, one it refuses, and halving
until they meet. Each probe is a real move of a few millimetres.

**This moves the 96-head**, forward along Y and then home in Y and Z when it parks. The deck has
to be clear along its sweep. It refuses to start unless the head is retracted to its safety height,
since Y travel happens at whatever Z the head is at. Gated on `allow_x_arm_move`.


In [25]:
if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
elif star.head96 is None:
  print("no 96-head on this machine")
else:
  head = star.head96
  # The head travels in Y at whatever Z it is at, so it must be retracted first: raised, it sweeps
  # over what is on the deck; low, it sweeps through it.
  z_now = await head.request_stop_disk_z()
  z_range = head.configuration.z_range
  if z_range is None:
    raise RuntimeError("the head's Z window was not probed; run setup before this")
  if z_now < z_range[1] - 1.0:
    raise RuntimeError(
      f"the head sits at z={z_now} mm, below its safety height of {z_range[1]} mm - retract it "
      "with head.move_to_z_safety() before sweeping it in Y"
    )
  print(f"head is retracted at z={z_now} mm")

  y_low, y_high = head.configuration.y_range
  print(f"documented travel: {y_low} to {y_high} mm")
  print(f"the drive reports it is at {await head.request_y_position():.2f} mm")

  accepted, refused = (y_low + y_high) / 2, y_low  # the middle works; the documented low does not
  await head.move_y(accepted)
  print(f"  {accepted:7.2f} mm accepted, drive reports {await head.request_y_position():7.2f} mm")

  for _ in range(8):
    if accepted - refused <= 0.5:
      break
    probe = (accepted + refused) / 2
    try:
      await head.move_y(probe)
    except Exception as error:  # noqa: BLE001 - the machine says what it will not do
      refused = probe
      print(
        f"  {probe:7.2f} mm refused, drive reports {await head.request_y_position():7.2f} mm"
        f"  ({error})"
      )
      continue
    accepted = probe
    print(f"  {probe:7.2f} mm accepted, drive reports {await head.request_y_position():7.2f} mm")

  print(f"\nthe permitted area starts between {refused:.2f} and {accepted:.2f} mm")
  print(f"the documented minimum is {y_low}, so {accepted - y_low:.2f} mm of it is out of reach")

  await head.park()
  print(f"\nparked; the drive reports {await head.request_y_position():.2f} mm")

2026-08-19 16:00:48,389 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RZid0847'
2026-08-19 16:00:48,399 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RZid0847rz+67401 +67393')
2026-08-19 16:00:48,402 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0848'
2026-08-19 16:00:48,412 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0848ry+28500 +28496')
2026-08-19 16:00:48,414 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0849rayv'


head is retracted at z=336.97 mm
documented travel: 93.75 to 562.5 mm
the drive reports it is at 445.25 mm


2026-08-19 16:00:48,438 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0849yv25000')
2026-08-19 16:00:48,442 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0850rayr'
2026-08-19 16:00:48,451 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0850yr35000')
2026-08-19 16:00:48,454 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0YAid0851ya21000yv25000yr35000yw15'
2026-08-19 16:00:49,465 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0YAid0851er00')
2026-08-19 16:00:49,470 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0852'
2026-08-19 16:00:49,479 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0852ry+21000 +21005')
2026-08-19 16:00:49,483 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0853'
2026-08-19 16:00:49,492 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0853ry+21000 +21005')
2026-08-19 16:00:49,496 - pylabrobot.io.usb - IO - [0x

   328.12 mm accepted, drive reports  328.20 mm


2026-08-19 16:00:50,530 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0YAid0856er00')
2026-08-19 16:00:50,535 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0857'
2026-08-19 16:00:50,545 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0857ry+13500 +13506')
2026-08-19 16:00:50,549 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0858'
2026-08-19 16:00:50,559 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0858ry+13500 +13506')
2026-08-19 16:00:50,563 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0859rayv'
2026-08-19 16:00:50,572 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0859yv25000')
2026-08-19 16:00:50,576 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0860rayr'
2026-08-19 16:00:50,586 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0860yr35000')
2026-08-19 16:00:50,589 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write

...

   104.74 mm accepted, drive reports  104.83 mm


2026-08-19 16:00:53,095 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0YAid0886er00')
2026-08-19 16:00:53,100 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0887'
2026-08-19 16:00:53,109 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0887ry+06586 +06590')
2026-08-19 16:00:53,112 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0888'
2026-08-19 16:00:53,122 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0888ry+06586 +06590')
2026-08-19 16:00:53,126 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0889rayv'
2026-08-19 16:00:53,135 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0889yv25000')
2026-08-19 16:00:53,138 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0890rayr'
2026-08-19 16:00:53,148 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0890yr35000')
2026-08-19 16:00:53,152 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write

   102.91 mm accepted, drive reports  102.97 mm
   101.99 mm refused, drive reports  102.97 mm  ({'CoRe 96 Head': UnknownHamiltonError('Y drive position outside of permitted area')}, H0YAid0891er58)

the permitted area starts between 101.99 and 102.91 mm
the documented minimum is 93.75, so 9.16 mm of it is out of reach


2026-08-19 16:00:55,245 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0MOid0894er00')
2026-08-19 16:00:55,250 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0895'
2026-08-19 16:00:55,260 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0895ry+35485 +35482')
2026-08-19 16:00:55,263 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0896'
2026-08-19 16:00:55,273 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0896ry+35485 +35482')



parked; the drive reports 554.41 mm


## 13g- Is it ringing, or still arriving?

Every first read after a move sits at or below the target, and the deficit grows with acceleration:
nothing at index 1, up to 0.3 mm at index 5. A second later every one of them reads exactly on
target. So the arm gets there - the question is what it is doing in between.

Two mechanisms fit the summary and want different fixes.

- **Ringing.** The arm oscillates about the target and damps out. Sampling at an arbitrary moment
  should read high about as often as low, and waiting a fixed time is the fix.
- **Still arriving.** The drive closes the last increments slowly and never crosses the target.
  Then reading until two reads agree is the fix, and it may take far less than a second.

Nine first reads, none above the target, points at the second - but each cell was measured once,
and the differences are one or two increments against a read that jitters by one. So this repeats
each condition and samples the whole approach rather than one point of it.

- **Repeats** give the front-versus-back difference an error bar, rather than one number each.
- **Fast polling** shows the shape: an approach that creeps up to the target, or one that crosses
  it and comes back.
- **Both directions** separate travel from gravity: if the deficit always trails the direction of
  travel, the arm is arriving late, not sagging.

The head hangs to the left of the carriage - channel A1 sits 368.2 mm from its centre - so the mass
on the arm is off-centre, and a wobble it excites need not be symmetric. That shows up here as first
errors that do not mirror: a large deficit moving one way and a small one moving the other, rather
than equal and opposite.

One thing no repeat can settle: every position here is the carriage encoder, on the drive. The head
swinging about the carriage while the carriage stands still is invisible to `X0 RX` and to `C0 RX`
alike. So this measures when the carriage has arrived, not when the head has.

**This moves the arm and the 96-head.** Gated on `allow_x_arm_move`.


In [26]:
import statistics
import time

repeats = 5
poll_for = 0.4  # seconds of fast reading after each move
target = 600.0

if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
elif star.head96 is None:
  print("no 96-head on this machine")
else:
  head = star.head96
  y_low, y_high = head.configuration.y_range
  span = y_high - y_low
  head_positions = {"front": y_low + span / 4, "back": y_high - span / 4}

  async def approach(from_x: float, acceleration: int) -> list:
    """Move to the target from `from_x` and read as fast as the link allows. Returns (ms, mm)."""
    await star.x_arm.move_x(from_x)
    await star.driver.send_command(
      module="X0", command="XP", la=f"{round(target * 10):05}", lr=f"{acceleration:01}", lw="7"
    )
    started = time.monotonic()
    series = []
    while time.monotonic() - started < poll_for:
      series.append(((time.monotonic() - started) * 1000, await star.x_arm.request_position()))
    return series

  print(
    f"{'head':>6} {'accel':>6} {'from':>6} {'first':>8} {'settled':>8} {'crossed?':>9} {'ms to settle':>13}"
  )
  for where, y in head_positions.items():
    await head.move_y(y)
    for acceleration in (3, 5):
      for direction, from_x in (("below", target - 100.0), ("above", target + 100.0)):
        firsts, settle_times, crossings = [], [], 0
        for _ in range(repeats):
          series = await approach(from_x, acceleration)
          firsts.append(series[0][1] - target)
          # any reading past the target, in the direction of travel, means it crossed
          if direction == "below" and max(mm for _, mm in series) > target + 0.05:
            crossings += 1
          if direction == "above" and min(mm for _, mm in series) < target - 0.05:
            crossings += 1
          settled = next((ms for ms, mm in series if abs(mm - target) <= 0.05), float("nan"))
          settle_times.append(settled)
        print(
          f"{where:>6} {acceleration:>6} {direction:>6}"
          f" {statistics.mean(firsts):+8.2f} {statistics.pstdev(firsts):8.2f}"
          f" {crossings:>4}/{repeats:<4}"
          f" {statistics.mean([t for t in settle_times if t == t]):13.0f}"
        )
  print(
    "\nfirst = mean error at the first read with its spread, in mm;"
    " crossed = runs that went past the target;"
    " ms to settle = when the reading first came within 0.05 mm"
  )
  await head.park()

2026-08-19 16:00:55,321 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0897rayv'
2026-08-19 16:00:55,331 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0897yv25000')
2026-08-19 16:00:55,335 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0898rayr'
2026-08-19 16:00:55,345 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0898yr35000')
2026-08-19 16:00:55,347 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0YAid0899ya13500yv25000yr35000yw15'


  head  accel   from    first  settled  crossed?  ms to settle


2026-08-19 16:00:57,011 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0YAid0899er00')
2026-08-19 16:00:57,017 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0900'
2026-08-19 16:00:57,025 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0900ry+13500 +13506')
2026-08-19 16:00:57,029 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0901la05000lr3lw7'
2026-08-19 16:00:58,240 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0901er00')
2026-08-19 16:00:58,245 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0902'
2026-08-19 16:00:58,254 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0902rx+05002 +000050020')
2026-08-19 16:00:58,257 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.2 mm
2026-08-19 16:00:58,260 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0903la06000lr3lw7'
2026-08-19 16:00

 front      3  below    -0.20     0.00    0/5               51


2026-08-19 16:01:12,511 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid1072er00')
2026-08-19 16:01:12,516 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid1073'
2026-08-19 16:01:12,525 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid1073rx+06998 +000069979')
2026-08-19 16:01:12,529 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 700.0 mm and came to rest at 699.8 mm
2026-08-19 16:01:12,531 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid1074la06000lr3lw7'
2026-08-19 16:01:13,741 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid1074er00')
2026-08-19 16:01:13,745 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid1075'
2026-08-19 16:01:13,755 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid1075rx+06002 +000060019')
2026-08-19 16:01:13,759 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid1076'
2026-08-19 16:01:13,768 -

...

2026-08-19 16:02:23,263 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid1933er00')
2026-08-19 16:02:23,268 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid1934'
2026-08-19 16:02:23,279 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid1934rx+05001 +000050010')
2026-08-19 16:02:23,281 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.1 mm
2026-08-19 16:02:23,284 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid1935la06000lr5lw7'
2026-08-19 16:02:24,297 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid1935er00')
2026-08-19 16:02:24,302 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid1936'
2026-08-19 16:02:24,314 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid1936rx+05997 +000059969')
2026-08-19 16:02:24,317 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid1937'
2026-08-19 16:02:24,327 -

  back      5  below    -0.32     0.04    5/5               90


2026-08-19 16:02:36,520 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid2103er00')
2026-08-19 16:02:36,525 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid2104'
2026-08-19 16:02:36,535 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid2104rx+06999 +000069985')
2026-08-19 16:02:36,538 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 700.0 mm and came to rest at 699.9 mm
2026-08-19 16:02:36,540 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid2105la06000lr5lw7'
2026-08-19 16:02:37,549 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid2105er00')
2026-08-19 16:02:37,553 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid2106'
2026-08-19 16:02:37,563 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid2106rx+06003 +000060034')
2026-08-19 16:02:37,567 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid2107'
2026-08-19 16:02:37,577 -

  back      5  above    +0.32     0.04    5/5               57

first = mean error at the first read with its spread, in mm; crossed = runs that went past the target; ms to settle = when the reading first came within 0.05 mm


2026-08-19 16:02:49,675 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0MOid2275er00')
2026-08-19 16:02:49,680 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid2276'
2026-08-19 16:02:49,689 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid2276ry+35485 +35481')


## 14- The instrument configuration, and what writing it would mean

`C0 AK` writes the machine's non-volatile configuration. It takes **21 parameters**, each with a
default, and the master's convention is that an unsent parameter takes its default - so a partial
`AK` does not change one field, it rewrites all of them. Sending `kb` alone would declare no
channels, no 96-head, a different arm width and a different waste position.

Every one of the 21 is readable: `C0 RM` answers `kb` and `kp`, `C0 QM` the other 19. The cell below
reads them, rebuilds the command that would restore exactly what the machine says today, and shows
what changes if the front cover monitoring bit is set. **It sends nothing.**

Why we would want to: with `kb` bit 2 clear, `C0 QC` answered `qc1` with the cover open, so the
master is not reading the switch. Setting the bit is the only way to find out whether `QC` reports
the cover on a machine that declares the monitoring - and it is also what makes the machine abort a
run when the cover opens, which is why it was turned off in the first place.


In [27]:
# The 21 parameters AK takes, in the order the specification lists them.
AK_PARAMETERS = "ka ke xt xa xw kb xl xn xr xo xm xx xu xv kp ys kl km ym yu yx".split()


def read_fields(reply: str) -> dict:
  """The two-letter fields in a reply, as the machine wrote them."""
  return dict(re.findall(r"([a-z]{2})([0-9A-Fa-f]+)", reply.split("er00/00", 1)[-1]))


if protocol_mode != "execution":
  raise SystemExit("nothing to read: a simulated machine has no configuration to rebuild")

machine = await star.driver.send_command(module="C0", command="RM")
extended = await star.driver.send_command(module="C0", command="QM")
read = {**read_fields(extended), **read_fields(machine)}

missing = [name for name in AK_PARAMETERS if name not in read]
print(f"read {len(AK_PARAMETERS) - len(missing)} of {len(AK_PARAMETERS)} parameters")
if missing:
  print(f"MISSING, so a safe write is not possible: {missing}")
else:
  as_it_stands = "".join(f"{name}{read[name]}" for name in AK_PARAMETERS)
  print(f"\nrestores exactly what the machine says now:\n  C0AK{as_it_stands}")

  with_monitoring = dict(read)
  with_monitoring["kb"] = f"{int(read['kb'], 16) | 0b100:02X}"
  proposed = "".join(f"{name}{with_monitoring[name]}" for name in AK_PARAMETERS)
  print(f"\nwith the front cover monitoring bit set:\n  C0AK{proposed}")
  print(f"\nkb {read['kb']} -> {with_monitoring['kb']}")
  print(
    "everything else identical:",
    as_it_stands.replace(f"kb{read['kb']}", "")
    == proposed.replace(f"kb{with_monitoring['kb']}", ""),
  )

2026-08-19 16:02:49,719 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RMid2277'
2026-08-19 16:02:49,780 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RMid2277er00/00kb0Bkp08 C00000 X00000 P10000 P20000 P30000 P40000 P50000 P60000 P70000 P80000 I00000 R00000 H00000')
2026-08-19 16:02:49,784 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QMid2278'
2026-08-19 16:02:49,822 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QMid2278er00/00ka010003xt54xa54xw13400xl07xr00xm03500xx11400ys090xu3540xv3700yu0060kl360kc0yx0060ke00000000xn00xo00ym6065kr0km360')


read 21 of 21 parameters

restores exactly what the machine says now:
  C0AKka010003ke00000000xt54xa54xw13400kb0Bxl07xn00xr00xo00xm03500xx11400xu3540xv3700kp08ys090kl360km360ym6065yu0060yx0060

with the front cover monitoring bit set:
  C0AKka010003ke00000000xt54xa54xw13400kb0Fxl07xn00xr00xo00xm03500xx11400xu3540xv3700kp08ys090kl360km360ym6065yu0060yx0060

kb 0B -> 0F
everything else identical: True


### Writing it

Only with `allow_configuration_write = True`, set in the cell itself so it cannot be reached by
running the notebook top to bottom. It writes the full command built above, reads the configuration
back, and prints the restore command in case the read-back does not match.

Keep the restore line from the cell above. If anything goes wrong, sending it puts the machine back.


In [28]:
allow_configuration_write = False

if not allow_configuration_write:
  print("skipped. this rewrites the machine's non-volatile configuration")
elif missing:
  print("refused: not every parameter could be read")
else:
  print(f"restore command, keep this:\n  C0AK{as_it_stands}\n")
  print(await star.driver.send_raw_command(f"C0AK{proposed}"))

  after = {
    **read_fields(await star.driver.send_command(module="C0", command="QM")),
    **read_fields(await star.driver.send_command(module="C0", command="RM")),
  }
  for name in AK_PARAMETERS:
    if after.get(name) != with_monitoring.get(name):
      print(f"  {name}: wrote {with_monitoring.get(name)}, reads back {after.get(name)}")
  print(
    "read back identical to what was written:",
    all(after.get(n) == with_monitoring.get(n) for n in AK_PARAMETERS),
  )

skipped. this rewrites the machine's non-volatile configuration


## 15- Raw command escape hatch

Anything not yet wrapped in a named method can be sent directly. **Only send commands you have
confirmed are read-only** - this bypasses every guard in the driver.

In [29]:
# C0 RF - request the master's firmware version. Read-only.
print(await star.driver.send_command(module="C0", command="RF"))

# the same thing as a raw string, id included
# print(await star.driver.send_raw_command("C0RFid9999"))

# Read the autoload's stored configuration, whose first field is the scanner X-drive resolution:
# 0 => 0.1 mm/step, 1 => 0.125 mm/step on a pilot-lot unit. The driver hardcodes 0.1, which is
# only right for the units that have it.
#
# Two candidates, because the read differs by autoload generation: `QU` on the later firmware,
# and the generic parameter read `RA` on the generation this machine reports. Both are
# read-only. Whichever answers, its reply is the shape a
# request_x_resolution() would parse - so print it raw.
for attempt in ("I0QUid9990", "I0RAid9991raau"):
  try:
    print(attempt, "->", await star.driver.send_raw_command(attempt))
  except Exception as e:  # noqa: BLE001 - whatever the machine says is the answer
    print(attempt, "-> refused:", e)

2026-08-19 16:02:49,858 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RFid2279'
2026-08-19 16:02:49,874 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RFid2279er00/00rf7.6S 25 2021_11_05 (GRU C0)')
2026-08-19 16:02:49,878 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0QUid9990'
2026-08-19 16:02:49,887 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0QUid9990er30')
2026-08-19 16:02:49,890 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RAid9991raau'
2026-08-19 16:02:49,901 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RAid9991au0 0 0 0 0')


C0RFid2279er00/00rf7.6S 25 2021_11_05 (GRU C0)
I0QUid9990 -> refused: {'Auto Load': UnknownHamiltonError('Unknown command')}, I0QUid9990er30
I0RAid9991raau -> I0RAid9991au0 0 0 0 0


## 16- What is ported, and what is not

Every module hangs off the same reply router, so each remaining one is a module to add rather
than new plumbing.

| Module | Node | State |
|---|---|---|
| Master | `C0` | configuration, initialization, tip presence |
| Pipetting channels | `P1`-`PG` | firmware, width, installed hardware, initialization |
| X-drives | `X0` | firmware, absolute move |
| 96-head | `H0` | firmware, hardware, drive parameters, retract, initialization |
| iSWAP | `R0` | not ported |
| Autoload | `I0` | not ported |
| Wash stations, pumps | `W1`/`W2`, `HW`/`HU`/`HV` | not ported |

On a machine with a 96-head, setup retracts it to Z safety and probes how far it reaches. If the
head reports itself uninitialized, setup says so rather than guessing: initializing it ejects
whatever is mounted, so it needs the position to eject at - `head96.initialize(x, y, z)`.

## 17- Teardown

In [30]:
await star.stop()
print("disconnected. connected:", star.driver.connected, "| setup done:", star.driver.setup_done)

# The log is append-only and stays open for the rest of the session - nothing to close.

2026-08-19 16:02:49,913 - pylabrobot.io.usb - WARNING - Closing connection to USB device.


disconnected. connected: False | setup done: False
